# Check training image files — d2 full dataset

## Goal

Confirm that every `image_name` in `train.csv` has:

- a JPEG image in `data/raw/jpeg/train` (`.jpg` or `.jpeg`), and
- a DICOM image in `data/raw/train` (`.dcm`).

The reusable validation logic lives in `melanoma_detection.image_validation`. This notebook only supplies local paths and presents the results; it does not modify raw files.


## Setup

Set `RAW_DATA_DIR` to your local raw-data directory.


In [1]:
from pathlib import Path

from melanoma_detection.image_validation import check_image_inventory


# Change this one path to your local raw-data directory.
RAW_DATA_DIR = Path(
    r"C:/Users/thoma/Documents/GitHub/ADS2002-Melanoma/data/raw"
)

TRAIN_CSV = RAW_DATA_DIR / "train.csv"
JPEG_TRAIN_DIR = RAW_DATA_DIR / "jpeg" / "train"
DICOM_TRAIN_DIR = RAW_DATA_DIR / "train"

print(f"CSV:   {TRAIN_CSV}")
print(f"JPEG:  {JPEG_TRAIN_DIR}")
print(f"DICOM: {DICOM_TRAIN_DIR}")


CSV:   C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\raw\train.csv
JPEG:  C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\raw\jpeg\train
DICOM: C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma\data\raw\train


## Run the reusable inventory check


In [2]:
report = check_image_inventory(
    train_csv=TRAIN_CSV,
    jpeg_directory=JPEG_TRAIN_DIR,
    dicom_directory=DICOM_TRAIN_DIR,
)


## Results

### Summary counts


In [3]:
print(f"{'Check':<32} {'Count':>8}")
print("-" * 41)
for check_name, count in report.summary_rows():
    print(f"{check_name:<32} {count:>8,}")


Check                               Count
-----------------------------------------
CSV rows                           33,126
unique CSV image names             33,126
JPEG files                         33,126
DICOM files                        33,126
duplicated CSV image names              0
CSV names missing JPEG                  0
CSV names missing DICOM                 0
JPEG files absent from CSV              0
DICOM files absent from CSV             0
duplicate JPEG stems                    0
duplicate DICOM stems                   0


### Inspect mismatches

Only the first 20 identifiers in each category are displayed so output remains manageable.


In [4]:
for category, examples in report.mismatch_examples(limit=20).items():
    print(f"{category} ({len(examples)} shown): {list(examples)}")


duplicate_csv_names (0 shown): []
missing_jpeg (0 shown): []
missing_dicom (0 shown): []
orphan_jpeg (0 shown): []
orphan_dicom (0 shown): []
duplicate_jpeg_stems (0 shown): []
duplicate_dicom_stems (0 shown): []


## Final checks


In [5]:
print(
    "Every CSV image has JPEG and DICOM:",
    report.all_csv_rows_have_both_formats,
)
print("Directories exactly match train.csv:", report.directories_exactly_match_csv)

# strict=True also rejects duplicate identifiers/stems and orphan image files.
report.assert_complete(strict=True)
print("All image-inventory checks passed.")


Every CSV image has JPEG and DICOM: True
Directories exactly match train.csv: True
All image-inventory checks passed.


## Reuse from another notebook

```python
from melanoma_detection.image_validation import check_image_inventory

report = check_image_inventory(TRAIN_CSV, JPEG_TRAIN_DIR, DICOM_TRAIN_DIR)
report.assert_complete(strict=True)
```

## Next steps

This validates filenames and one-to-one inventory coverage. A separate optional check could attempt to decode every JPEG and DICOM to identify corrupted file contents.
